In [1]:
import tarfile
import json
import torch
import re
import gc
import hashlib
import ast

import networkx as nx
import pandas as pd

from collections import defaultdict, Counter
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Load the exact same source file used in the graph builder
# (Change the filename here depending on which KG model you are baselining against)
file_path = "../kg_construction/subgraphs_qwen_structured_isco.xlsx"
df_base = pd.read_excel(file_path)

# Filter for users with at least one positive response
non_zero = df_base.groupby("cvid")["response"].sum() > 0
non_zero = set(non_zero[non_zero == True].index)
df_base = df_base[df_base["cvid"].isin(non_zero)]

# Enforce the strict hits/misses intersection from the graph builder
hits = set(df_base[df_base["response"] == 1]["cvid"])
misses = set(df_base[df_base["response"] == 0]["cvid"])
valid_users = hits.intersection(misses)

df_graphs = df_base[df_base["cvid"].isin(valid_users)].copy()

# Clean and align column names for downstream merging
df_graphs.rename(columns={'vacancy': 'humanjobid'}, inplace=True)

# Ensure humanjobid is a clean string (removing any .0 floats)
df_graphs['humanjobid'] = df_graphs['humanjobid'].astype(str).str.split('.').str[0]

print(f"Total valid interactions retained: {len(df_graphs)}")
df_graphs.head()

Total valid interactions retained: 7677


,cvid,humanjobid,response,graph,path_exists,num_nodes,num_edges,density,avg_degree,avg_clustering,shortest_path_length
0,1ea333bcf408489d88d6bb4ecad3852d,1527549,0,"{'directed': True, 'multigraph': False, 'graph...",True,6,6,0.400000,2.0,0.388889,4
3,6e5a9a5cac7e454bb17c85853850b108,1527507,0,"{'directed': True, 'multigraph': False, 'graph...",True,8,8,0.285714,2.0,0.291667,6
4,2772752bd7fb4fdebd3c2cdbf7539ee7,1527507,1,"{'directed': True, 'multigraph': False, 'graph...",True,8,8,0.285714,2.0,0.291667,6
5,d397d2f211854740ac73515be119150d,1527507,0,"{'directed': True, 'multigraph': False, 'graph...",True,8,8,0.285714,2.0,0.291667,6
10,29c6a2ea76834acebd79091f02b4888c,1527480,0,NaN,False,0,0,0.000000,0.0,0.000000,-1


In [3]:
texts = defaultdict(list)

with open("../../dataset/final_dataset/jobs.json", 'r') as f:
    data = json.load(f)

for item in data:
    texts["id"].append(item.get("humanjobid", ""))
    texts["company"].append(item.get("companytext", ""))
    texts["job title"].append(item.get("jobtitle", ""))
    texts["text"].append(item.get("searchtext", ""))

df = pd.DataFrame(texts)
df.head()

,id,company,job title,text
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...


In [4]:
with open("../outputs/final_outputs/anon_cvs_coalesced.json", 'r', encoding="utf-8") as f:
    data = json.load(f)

cvs = defaultdict(list)

for candidate in data:
    cvs["cvid"].append(candidate["cvid"])
    cvs["text"].append({k: str(v) for k, v in candidate.items() if k != "cvid"})

df_cv = pd.DataFrame(cvs)
df_cv.head()

,cvid,text
0,ebf358ed252344af8d1dd3f4c8516cbf,{'headline': 'Erfaren Senior IT SystemKonsulen...
1,6c6e7a728dcc4ade98e1367f4efe6fb7,"{'headline': 'Systemudvikler, support, Sql, Or..."
2,b1afb88436354245a405bcf36f4030c8,"{'headline': 'Datalog, full stack udvikler/lea..."
3,e9e192123ed44166993497619c499bd5,{'headline': 'Økonom-generalist-sagsbehandling...
4,46353e9962684179846f77175feebef9,{'headline': 'Skilled .NET Backend / Integrati...


In [5]:
# 1. Exact Split Logic
def get_split(user_id, train_size=0.8, val_size=0.1):
    hash_val = int(hashlib.md5(str(user_id).encode('utf-8')).hexdigest(), 16)
    bucket = (hash_val % 1000) / 1000.0  
    if bucket < train_size: return "train"
    elif bucket < (train_size + val_size): return "val"
    else: return "test"

# 2 & 3. Build the Target List natively from the DataFrame
file_path = "../kg_construction/subgraphs_qwen_structured_isco.xlsx"
df_base = pd.read_excel(file_path)

# Filter for users with at least one positive response
non_zero = df_base.groupby("cvid")["response"].sum() > 0
non_zero = set(non_zero[non_zero == True].index)
df_base = df_base[df_base["cvid"].isin(non_zero)]

# Enforce the strict hits/misses intersection from the graph builder
hits = set(df_base[df_base["response"] == 1]["cvid"])
misses = set(df_base[df_base["response"] == 0]["cvid"])
valid_users = hits.intersection(misses)
df_graphs = df_base[df_base["cvid"].isin(valid_users)].copy()

# Ensure standard naming and clean IDs
if 'vacancy' in df_graphs.columns:
    df_graphs.rename(columns={'vacancy': 'humanjobid'}, inplace=True)
df_graphs['humanjobid'] = df_graphs['humanjobid'].astype(str).str.split('.').str[0]

# Apply the topological constraint using the pre-calculated 'path_exists' column
df_target = df_graphs[df_graphs['path_exists'] == True][['cvid', 'humanjobid', 'response']].copy()

print(f"Absolutely verified graph-surviving pairs: {len(df_target)}")

# 4. Safely Merge Text (No Cartesian Explosions)
df['id'] = df['id'].astype(str)
df_cv['cvid'] = df_cv['cvid'].astype(str)

# CRITICAL: Drop duplicate IDs from text sources before merging
df_texts = df.drop_duplicates(subset=['id'])[['id', 'text']].copy()
df_cv_texts = df_cv.drop_duplicates(subset=['cvid'])[['cvid', 'text']].copy()

merged_df = df_target.merge(df_texts, left_on='humanjobid', right_on='id', how='inner')
merged_df.rename(columns={'text': 'vacancy_text'}, inplace=True)

merged_df = merged_df.merge(df_cv_texts, on='cvid', how='inner')
merged_df.rename(columns={'text': 'cv_text'}, inplace=True)
merged_df['split'] = merged_df['cvid'].apply(get_split)

# Overwrite df_target to ONLY contain edges that survived the text merge. 
df_target = merged_df[['cvid', 'humanjobid', 'response', 'split']].copy()
print(f"Absolutely verified mutually-surviving pairs: {len(df_target)}")

class TextRankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        print("Pre-tokenizing dataset (this takes a minute)...")
        
        cv_texts = dataframe['cv_text'].astype(str).tolist()
        vac_texts = dataframe['vacancy_text'].astype(str).tolist()
        
        # Tokenize everything upfront
        self.cv_encodings = tokenizer(
            cv_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )
        self.vac_encodings = tokenizer(
            vac_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )

        self.labels = dataframe['response'].values
        self.cvids = dataframe['cvid'].astype(str).tolist()
        self.vacancy_ids = dataframe['humanjobid'].astype(str).tolist()
        
        # --- THE CORE DIFFERENCE: GROUP BY CANDIDATE ---
        self.unique_cvids = dataframe['cvid'].unique().tolist()
        self.grouped_indices = dataframe.groupby('cvid').indices

    def __len__(self):
        # Length is now the number of unique candidates, NOT the number of total rows
        return len(self.unique_cvids)

    def __getitem__(self, idx):
        # 1. Look up the candidate
        cvid = self.unique_cvids[idx]
        # 2. Get the specific row indices for all N of their vacancies
        indices = self.grouped_indices[cvid] 
        
        # 3. Return the N-sized chunk of tensors for this single candidate
        return {
            'cv_input_ids': self.cv_encodings['input_ids'][indices],
            'cv_attention_mask': self.cv_encodings['attention_mask'][indices],
            'vac_input_ids': self.vac_encodings['input_ids'][indices],
            'vac_attention_mask': self.vac_encodings['attention_mask'][indices],
            'labels': torch.tensor(self.labels[indices], dtype=torch.float32),
            'cvid': [self.cvids[i] for i in indices],
            'vacancy_id': [self.vacancy_ids[i] for i in indices]
        }

def ranking_collate_fn(batch):
    """
    Takes a list of candidate dictionaries (where each dict contains N items)
    and concatenates them along the 0th dimension to mimic PyG batching.
    """
    return {
        'cv_input_ids': torch.cat([b['cv_input_ids'] for b in batch], dim=0),
        'cv_attention_mask': torch.cat([b['cv_attention_mask'] for b in batch], dim=0),
        'vac_input_ids': torch.cat([b['vac_input_ids'] for b in batch], dim=0),
        'vac_attention_mask': torch.cat([b['vac_attention_mask'] for b in batch], dim=0),
        'labels': torch.cat([b['labels'] for b in batch], dim=0),
        
        # Flatten the nested lists of strings
        'cvid': [c for b in batch for c in b['cvid']],
        'vacancy_id': [v for b in batch for v in b['vacancy_id']]
    }

tokenizer = AutoTokenizer.from_pretrained("jjzha/dajobbert-base-uncased")

setting = "sentence"

if setting == "sentence":
    train_df = merged_df[(merged_df['split'] == 'train') & (merged_df['response'] == 1.0)]
    train_dataset = TextRankingDataset(train_df, tokenizer)
else:
    train_dataset = TextRankingDataset(merged_df[merged_df['split'] == 'train'], tokenizer)

val_dataset = TextRankingDataset(merged_df[merged_df['split'] == 'val'], tokenizer)
test_dataset = TextRankingDataset(merged_df[merged_df['split'] == 'test'], tokenizer)

# Use batch_size=16 for train (16 candidates flattened together).
# Use batch_size=1 for val/test to guarantee exactly 1 candidate is evaluated per batch.
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, pin_memory=True, collate_fn=ranking_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, pin_memory=True, collate_fn=ranking_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, pin_memory=True, collate_fn=ranking_collate_fn)

print(f"Train candidates: {len(train_dataset)} | Val candidates: {len(val_dataset)} | Test candidates: {len(test_dataset)}")

Absolutely verified graph-surviving pairs: 6815
Absolutely verified mutually-surviving pairs: 6815
Pre-tokenizing dataset (this takes a minute)...
Pre-tokenizing dataset (this takes a minute)...
Pre-tokenizing dataset (this takes a minute)...
Train candidates: 276 | Val candidates: 41 | Test candidates: 37


In [6]:
json_result = (
    merged_df[merged_df["split"] == "test"]
    .groupby("cvid")["humanjobid"]
    .agg(list)
    .to_json()
)

with open("test_set_sample.json", "w") as f:
    json.dump(json_result, f)

In [7]:
if setting == "sentence":
    train_path = f"../dataloaders/sentence_trainloader.pth"
    val_path = f"../dataloaders/sentence_valloader.pth"
    test_path = f"../dataloaders/sentence_testloader.pth"
else:
    train_path = f"../dataloaders/text_trainloader.pth"
    val_path = f"../dataloaders/text_valloader.pth"
    test_path = f"../dataloaders/text_testloader.pth"

torch.save(train_loader, train_path)
torch.save(val_loader, val_path)
torch.save(test_loader, test_path)

In [8]:
def flatten_features(feat):
    """Recursively flattens lists/tuples to match flat tensor dimensions."""
    flat_list = []
    for item in feat:
        if isinstance(item, (list, tuple)):
            flat_list.extend(flatten_features(item))
        else:
            flat_list.append(item)
    return flat_list

def verify_dataloader_alignment(graph_loader, text_loader, split_name="Validation"):
    print(f"--- Verifying {split_name} Split ---")
    
    graph_pairs = Counter()
    for batch in graph_loader:
        labels = batch.y.view(-1).tolist()
        
        # Safely flatten the PyG lists-of-lists down to 1D
        cvids = flatten_features(batch["cvid"])
        vacs = flatten_features(batch["vacancy_id"])
            
        for c, v, l in zip(cvids, vacs, labels):
            c_str = str(c.item() if hasattr(c, 'item') else c).strip()
            v_str = str(v.item() if hasattr(v, 'item') else v).strip()
            graph_pairs[(c_str, v_str, float(l))] += 1
            
    text_pairs = Counter()
    for batch in text_loader:
        labels = batch['labels'].view(-1).tolist()
        
        # Safely flatten the Text Dataloader lists down to 1D
        cvids = flatten_features(batch["cvid"])
        vacs = flatten_features(batch["vacancy_id"])
        
        for c, v, l in zip(cvids, vacs, labels):
            c_str = str(c.item() if hasattr(c, 'item') else c).strip()
            v_str = str(v.item() if hasattr(v, 'item') else v).strip()
            text_pairs[(c_str, v_str, float(l))] += 1
            
    # 1. Check absolute lengths (including duplicates if any exist)
    g_len = sum(graph_pairs.values())
    t_len = sum(text_pairs.values())
    print(f"Graph pairs: {g_len} | Text pairs: {t_len}")
    
    if g_len != t_len:
        print("❌ ERROR: DataLoaders have different total numbers of pairs!")
        
    # 2. Check exact pair mapping alignment
    missing_in_text = graph_pairs - text_pairs
    missing_in_graph = text_pairs - graph_pairs
    
    if not missing_in_text and not missing_in_graph:
        positives = sum(count for pair, count in graph_pairs.items() if pair[2] == 1.0)
        print(f"✅ Alignment Confirmed! Both contain exactly {positives} positive edges and {g_len - positives} negative edges mapped to the exact same candidates and vacancies.")
        return True
    else:
        print("❌ ERROR: Pair mismatches detected!")
        if missing_in_text:
            diff_count = sum(missing_in_text.values())
            print(f"   -> {diff_count} pairs found in Graph but missing/mismatched in Text.")
            # Print a few examples for debugging
            for pair in list(missing_in_text.keys())[:3]:
                print(f"      Example missing in text: {pair}")
                
        if missing_in_graph:
            diff_count = sum(missing_in_graph.values())
            print(f"   -> {diff_count} pairs found in Text but missing/mismatched in Graph.")
            for pair in list(missing_in_graph.keys())[:3]:
                print(f"      Example missing in graph: {pair}")
        return False

trainloader = torch.load(f'../dataloaders/graph_trainloader_qwen_structured_isco.pth',
                         weights_only=False)
valloader = torch.load(f'../dataloaders/graph_valloader_qwen_structured_isco.pth',
                         weights_only=False)
testloader = torch.load(f'../dataloaders/graph_testloader_qwen_structured_isco.pth',
                         weights_only=False)

if setting != "sentence":
    verify_dataloader_alignment(trainloader, train_loader, "Train")
    verify_dataloader_alignment(valloader, val_loader, "Validation")
    verify_dataloader_alignment(testloader, test_loader, "Test")